In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as tt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchsummary


In [2]:
!pip install -U ultralytics

from ultralytics import YOLO


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
!pip install kagglehub
import kagglehub

path = kagglehub.dataset_download(
    "javiersanchezsoriano/roundabout-aerial-images-for-vehicle-detection"
)

print("Dataset downloaded to:", path)

!ls -R "$path" | head -50

100%|██████████| 12.4G/12.4G [02:42<00:00, 82.0MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/javiersanchezsoriano/roundabout-aerial-images-for-vehicle-detection/versions/2
/root/.cache/kagglehub/datasets/javiersanchezsoriano/roundabout-aerial-images-for-vehicle-detection/versions/2:
data.csv
original
roundabouts.csv

/root/.cache/kagglehub/datasets/javiersanchezsoriano/roundabout-aerial-images-for-vehicle-detection/versions/2/original:
original

/root/.cache/kagglehub/datasets/javiersanchezsoriano/roundabout-aerial-images-for-vehicle-detection/versions/2/original/original:
annotations
imgs

/root/.cache/kagglehub/datasets/javiersanchezsoriano/roundabout-aerial-images-for-vehicle-detection/versions/2/original/original/annotations:
00001_frame000000_original.xml
00001_frame000001_original.xml
00001_frame000002_original.xml
00001_frame000003_original.xml
00001_frame000004_original.xml
00001_frame000005_original.xml
00001_frame000006_original.xml
00001_frame000007_original.xml
00001_frame000008_original.xml
00001_frame000009_o

In [4]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
from sklearn.model_selection import train_test_split
import shutil

# ---- paths ----
DATASET_ROOT = path  # from kagglehub.dataset_download()
IMG_DIR = f"{DATASET_ROOT}/original/original/imgs"
ANN_DIR = f"{DATASET_ROOT}/original/original/annotations"

OUT_ROOT = "/content/roundabout_yolo"

# ---- classes ----
CLASSES = ["vehicle", "car", "truck", "bus"]

# ---- output dirs ----
for split in ["train", "val"]:
    os.makedirs(f"{OUT_ROOT}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUT_ROOT}/labels/{split}", exist_ok=True)

def convert_bbox(size, box):
    w, h = size
    xmin, xmax, ymin, ymax = box
    x = (xmin + xmax) / 2 / w
    y = (ymin + ymax) / 2 / h
    bw = (xmax - xmin) / w
    bh = (ymax - ymin) / h
    return x, y, bw, bh

# ---- split ----
images = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".jpg")])
train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

def process_split(img_list, split):
    for img_name in img_list:
        xml_file = f"{ANN_DIR}/{Path(img_name).stem}.xml"
        tree = ET.parse(xml_file)
        root = tree.getroot()

        w = int(root.find("size/width").text)
        h = int(root.find("size/height").text)

        yolo_lines = []

        for obj in root.findall("object"):
            cls_name = obj.find("name").text.lower()
            if cls_name not in CLASSES:
                continue

            # merged class
            cls_id = 0
            box = obj.find("bndbox")

            xmin = int(box.find("xmin").text)
            xmax = int(box.find("xmax").text)
            ymin = int(box.find("ymin").text)
            ymax = int(box.find("ymax").text)

            bb = convert_bbox((w, h), (xmin, xmax, ymin, ymax))
            yolo_lines.append(
                f"{cls_id} " + " ".join(f"{x:.6f}" for x in bb)
            )

        # save label file
        with open(f"{OUT_ROOT}/labels/{split}/{Path(img_name).stem}.txt", "w") as f:
            f.write("\n".join(yolo_lines))

        shutil.copy(
            f"{IMG_DIR}/{img_name}",
            f"{OUT_ROOT}/images/{split}/{img_name}"
        )

# ---- run conversion ----
process_split(train_imgs, "train")
process_split(val_imgs, "val")

print("✅ YOLO conversion done (1 classes)")


✅ YOLO conversion done (1 classes)


In [5]:
data_yaml = f"""
path: {OUT_ROOT}
train: images/train
val: images/val

nc: 1
names: ["vehicle"]
"""

with open(f"{OUT_ROOT}/data.yaml", "w") as f:
    f.write(data_yaml)


In [6]:
model = YOLO("yolov8s.pt")  # load a pretrained model (recommended for training)


# Train the model
model.train(
    data="/content/roundabout_yolo/data.yaml",
    epochs=5,
    imgsz=1280,
    batch=8,
    close_mosaic=10,   # reduce mosaic later in training
    mosaic=0.2         # better for aerial scenes
)

Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/roundabout_yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.2, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a82d4368e00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [7]:
# Validate the model
metrics = model.val(split='val')  # dados do teste
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list containing mAP50-95 for each category

Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3185.7±547.5 MB/s, size: 903.4 KB)
val: Scanning /content/roundabout_yolo/labels/val.cache... 3095 images, 147 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3095/3095 1.3Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 194/194 2.0it/s 1:39
                   all       3095      48143      0.997      0.998      0.994      0.878
Speed: 4.0ms preprocess, 18.8ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/runs/detect/val


array([    0.87834])

In [8]:


# O YOLO guarda a média (Mean) destas métricas nos atributos .mp e .mr
precision = metrics.box.mp   # Mean Precision (média de todas as classes)
recall    = metrics.box.mr   # Mean Recall (média de todas as classes)

# O F1 é devolvido como um array (um valor por classe), por isso fazemos a média
f1_score  = metrics.box.f1.mean()

# 4. Imprimir tudo bonitinho
print(f"PRECISION: {precision:.4f}")
print(f"RECALL:    {recall:.4f}")
print(f"F1-SCORE:  {f1_score:.4f}")

PRECISION: 0.9968
RECALL:    0.9982
F1-SCORE:  0.9975


In [9]:
import shutil
from google.colab import files

# 1. Qual a pasta que queres descarregar?
# (Pode ser '/content/A2PartII_APA2025' ou '/content/runs')
pasta_para_baixar = '/content/runs/detect/val'

# 2. Nome do zip
nome_zip = '/content/test_yolo8s_pret'

# 3. Criar o zip
print("A comprimir...")
shutil.make_archive(nome_zip, 'zip', pasta_para_baixar)

# 4. Descarregar
print("A iniciar download...")
files.download(nome_zip + '.zip')

A comprimir...
A iniciar download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
from google.colab import files
import os

# Path to the checkpoint you want (adjust if you renamed the experiment)
ckpt_path = 'runs/detect/train/weights/best.pt'   # <-- change "exp" if needed

# Make sure the file exists
assert os.path.isfile(ckpt_path), f"{ckpt_path} not found!"

# Trigger the download dialog
files.download(ckpt_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
model.predict(
    source="video_000_crop.mp4",  # or frames folder
    conf=0.3,
    save_txt=True,       # <-- export predictions
    project="/content/yolo_preds",
    name="test_video"
)



WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/905) /content/video_000_crop.mp4: 1152x1280 5 vehicles, 49.0ms
video 1/1 (frame 2/905) /content/video_000_crop.mp4: 1152x1280 4 vehicles, 44.1ms
video 1/1 (frame 3/905) /content/video_000_crop.mp4: 1152x1280 5 vehicles, 44.1ms
video 1/1 (frame 4/905) /content/video_000_crop.mp4: 1152x1280 5 vehicles, 44.0ms
video 1/1 (frame 5/905) /content/video_000_crop.mp4: 1152x1280 4 vehicles, 39.1ms
video 1/1 (frame 6/905) /content/video_000_crop.

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'vehicle'}
 obb: None
 orig_img: array([[[ 42, 119, 118],
         [ 50, 127, 126],
         [ 46, 123, 122],
         ...,
         [ 87,  61,  48],
         [ 87,  60,  49],
         [ 87,  60,  49]],
 
        [[ 50, 127, 126],
         [ 54, 131, 130],
         [ 42, 119, 118],
         ...,
         [ 87,  61,  48],
         [ 87,  60,  49],
         [ 87,  60,  49]],
 
        [[ 50, 127, 126],
         [ 53, 130, 129],
         [ 43, 120, 119],
         ...,
         [ 87,  61,  48],
         [ 90,  63,  52],
         [ 91,  64,  53]],
 
        ...,
 
        [[ 96,  87,  84],
         [ 96,  87,  84],
         [ 95,  86,  83],
         ...,
         [  2,  40,  41],
         [ 16,  56,  52],
         [ 15,  55,  51]],
 
        [[ 95,  86,  83],
         [ 95,  86,  83],
         [ 94,  85,  82],
         ...,
         [  0,  3

In [13]:
!pip install ultralytics opencv-python-headless numpy
!pip install bytetrack-pytorch  # or your preferred ByteTrack repo


ERROR: Could not find a version that satisfies the requirement bytetrack-pytorch (from versions: none)
ERROR: No matching distribution found for bytetrack-pytorch


In [14]:
import cv2



In [ ]:
# Run tracking on a video
results = model.track(
    source='/content/video_002_crop.mp4',  # video file or folder of frames
    tracker='bytetrack.yaml',           # built-in tracker config
    persist=True,                       # keeps object IDs consistent
    save=True,                          # saves tracked video
    save_txt=True                       # optional: save tracked box coords
)



WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/19622) /content/video_002_crop.mp4: 1152x1280 7 vehicles, 52.9ms
video 1/1 (frame 2/19622) /content/video_002_crop.mp4: 1152x1280 6 vehicles, 45.0ms
video 1/1 (frame 3/19622) /content/video_002_crop.mp4: 1152x1280 6 vehicles, 44.9ms
video 1/1 (frame 4/19622) /content/video_002_crop.mp4: 1152x1280 6 vehicles, 45.0ms
video 1/1 (frame 5/19622) /content/video_002_crop.mp4: 1152x1280 6 vehicles, 42.4ms
video 1/1 (frame 6/19622) /content/vid